# Data

In [1]:
import os
import time
import pandas as pd
import requests
import yfinance as yf
from tqdm import tqdm

os.makedirs('data', exist_ok=True)

## Universe

In [2]:
stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'BRK-B', 'JNJ', 'JPM', 'XOM', 'BAC',
          'V', 'PFE', 'PG', 'CVX', 'HD', 'VZ', 'MA', 'WMT', 'CSCO', 'INTC',
          'KO', 'DIS', 'PEP', 'MRK', 'ABT', 'NKE', 'T', 'CMCSA', 'ORCL', 'IBM',
          'C', 'WFC', 'PM', 'GS', 'NFLX', 'BA', 'MCD', 'MMM', 'HON', 'AXP',
          'AVGO', 'UNP', 'UPS', 'ABBV', 'CAT', 'MS', 'CVS', 'UNH', 'LMT', 'TGT']

In [3]:
comps = {'AAPL': {'AAPL': ('Apple Inc.', 'Apple', 'Apple Inc.'),
                  'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                  'GOOGL': ('Alphabet Inc.', 'Google', 'Alphabet Inc.'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation')},
         'MSFT': {'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                  'AAPL': ('Apple Inc.', 'Apple', 'Apple Inc.'),
                  'GOOGL': ('Alphabet Inc.', 'Google', 'Alphabet Inc.'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation')},
         'GOOGL': {'GOOGL': ('Alphabet Inc.', 'Google', 'Alphabet Inc.'),
                   'AAPL': ('Apple Inc.', 'Apple', 'Apple Inc.'),
                   'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                   'NFLX': ('Netflix, Inc.', 'Netflix', 'Netflix'),
                   'META': ('Facebook, Inc.', 'Facebook', 'Facebook, Inc.')},
         'AMZN': {'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)'),
                  'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot'),
                  'TGT': ('Target Corporation', 'Target', 'Target Corporation'),
                  'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                  'DIS': ('The Walt Disney Company', 'Disney', 'The Walt Disney Company')},
         'JNJ': {'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer'),
                 'MRK': ('Merck & Co., Inc.', 'Merck', 'Merck & Co.'),
                 'ABT': ('Abbott Laboratories', 'Abbott', 'Abbott Laboratories'),
                 'ABBV': ('AbbVie Inc.', 'AbbVie', 'AbbVie')},
         'JPM': {'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                 'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                 'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                 'WFC': ('Wells Fargo & Company', 'Wells Fargo', 'Wells Fargo'),
                 'GS': ('The Goldman Sachs Group, Inc.', 'Goldman Sachs', 'Goldman Sachs')},
         'XOM': {'XOM': ('Exxon Mobil Corporation', 'Exxon Mobil', 'ExxonMobil'),
                 'CVX': ('Chevron Corporation', 'Chevron', 'Chevron Corporation'),
                 'COP': ('ConocoPhillips', 'ConocoPhillips', 'ConocoPhillips'),
                 'PSX': ('Phillips 66', 'Phillips 66', 'Phillips 66'),
                 'OXY': ('Occidental Petroleum Corporation', 'Occidental Petroleum', 'Occidental Petroleum')},
         'BAC': {'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                 'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                 'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                 'WFC': ('Wells Fargo & Company', 'Wells Fargo', 'Wells Fargo'),
                 'MS': ('Morgan Stanley', 'Morgan Stanley', 'Morgan Stanley')},
         'V': {'V': ('Visa Inc.', 'Visa', 'Visa Inc.'),
               'MA': ('Mastercard Incorporated', 'Mastercard', 'Mastercard'),
               'AXP': ('American Express Company', 'American Express', 'American Express'),
               'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
               'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America')},
         'PFE': {'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer'),
                 'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'MRK': ('Merck & Co., Inc.', 'Merck', 'Merck & Co.'),
                 'ABT': ('Abbott Laboratories', 'Abbott', 'Abbott Laboratories'),
                 'ABBV': ('AbbVie Inc.', 'AbbVie', 'AbbVie')},
         'PG': {'PG': ('The Procter & Gamble Company', 'Procter & Gamble', 'Procter & Gamble'),
                'KO': ('The Coca-Cola Company', 'Coca-Cola', 'The Coca-Cola Company'),
                'PEP': ('PepsiCo, Inc.', 'Pepsi', 'PepsiCo'),
                'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                'PM': ('Philip Morris International Inc.', 'Philip Morris', 'Philip Morris International')},
         'CVX': {'CVX': ('Chevron Corporation', 'Chevron', 'Chevron Corporation'),
                 'XOM': ('Exxon Mobil Corporation', 'Exxon Mobil', 'ExxonMobil'),
                 'COP': ('ConocoPhillips', 'ConocoPhillips', 'ConocoPhillips'),
                 'PSX': ('Phillips 66', 'Phillips 66', 'Phillips 66'),
                 'OXY': ('Occidental Petroleum Corporation', 'Occidental Petroleum', 'Occidental Petroleum')},
         'HD': {'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot'),
                'LOW': ("Lowe's Companies, Inc.", "Lowe's", "Lowe's"),
                'TGT': ('Target Corporation', 'Target', 'Target Corporation'),
                'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)')},
         'VZ': {'VZ': ('Verizon Communications Inc.', 'Verizon', 'Verizon Communications'),
                'T': ('AT&T Inc.', 'AT&T', 'AT&T'),
                'TMUS': ('T-Mobile US, Inc.', 'T-Mobile', 'T-Mobile US'),
                'CMCSA': ('Comcast Corporation', 'Comcast', 'Comcast'),
                'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)')},
         'MA': {'MA': ('Mastercard Incorporated', 'Mastercard', 'Mastercard'),
                'V': ('Visa Inc.', 'Visa', 'Visa Inc.'),
                'AXP': ('American Express Company', 'American Express', 'American Express'),
                'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America')},
         'WMT': {'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                 'TGT': ('Target Corporation', 'Target', 'Target Corporation'),
                 'COST': ('Costco Wholesale Corporation', 'Costco', 'Costco'),
                 'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)'),
                 'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot')},
         'CSCO': {'CSCO': ('Cisco Systems, Inc.', 'Cisco', 'Cisco Systems'),
                  'INTC': ('Intel Corporation', 'Intel', 'Intel'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation'),
                  'AVGO': ('Broadcom Inc.', 'Broadcom', 'Broadcom Inc.')},
         'INTC': {'INTC': ('Intel Corporation', 'Intel', 'Intel'),
                  'CSCO': ('Cisco Systems, Inc.', 'Cisco', 'Cisco Systems'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation'),
                  'AVGO': ('Broadcom Inc.', 'Broadcom', 'Broadcom Inc.')},
         'KO': {'KO': ('The Coca-Cola Company', 'Coca-Cola', 'The Coca-Cola Company'),
                'PEP': ('PepsiCo, Inc.', 'Pepsi', 'PepsiCo'),
                'MNST': ('Monster Beverage Corporation', 'Monster', 'Monster Beverage'),
                'PG': ('The Procter & Gamble Company', 'Procter & Gamble', 'Procter & Gamble'),
                'PM': ('Philip Morris International Inc.', 'Philip Morris', 'Philip Morris International')},
         'DIS': {'DIS': ('The Walt Disney Company', 'Disney', 'The Walt Disney Company'),
                 'CMCSA': ('Comcast Corporation', 'Comcast', 'Comcast'),
                 'NFLX': ('Netflix, Inc.', 'Netflix', 'Netflix'),
                 'T': ('AT&T Inc.', 'AT&T', 'AT&T'),
                 'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)')},
         'PEP': {'PEP': ('PepsiCo, Inc.', 'Pepsi', 'PepsiCo'),
                 'KO': ('The Coca-Cola Company', 'Coca-Cola', 'The Coca-Cola Company'),
                 'PG': ('The Procter & Gamble Company', 'Procter & Gamble', 'Procter & Gamble'),
                 'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                 'PM': ('Philip Morris International Inc.', 'Philip Morris', 'Philip Morris International')},
         'MRK': {'MRK': ('Merck & Co., Inc.', 'Merck', 'Merck & Co.'),
                 'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer'),
                 'ABT': ('Abbott Laboratories', 'Abbott', 'Abbott Laboratories'),
                 'ABBV': ('AbbVie Inc.', 'AbbVie', 'AbbVie')},
         'ABT': {'ABT': ('Abbott Laboratories', 'Abbott', 'Abbott Laboratories'),
                 'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer'),
                 'MRK': ('Merck & Co., Inc.', 'Merck', 'Merck & Co.'),
                 'ABBV': ('AbbVie Inc.', 'AbbVie', 'AbbVie')},
         'NKE': {'NKE': ('Nike, Inc.', 'Nike', 'Nike, Inc.'),
                 'ADDYY': ('Adidas AG', 'Adidas', 'Adidas'),
                 'UAA': ('Under Armour, Inc.', 'Under Armour', 'Under Armour'),
                 'MCD': ("McDonald's Corporation", "McDonald's", "McDonald's"),
                 'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot')},
         'T': {'T': ('AT&T Inc.', 'AT&T', 'AT&T'),
               'VZ': ('Verizon Communications Inc.', 'Verizon', 'Verizon Communications'),
               'TMUS': ('T-Mobile US, Inc.', 'T-Mobile', 'T-Mobile US'),
               'CMCSA': ('Comcast Corporation', 'Comcast', 'Comcast'),
               'DIS': ('The Walt Disney Company', 'Disney', 'The Walt Disney Company')},
         'CMCSA': {'CMCSA': ('Comcast Corporation', 'Comcast', 'Comcast'),
                   'DIS': ('The Walt Disney Company', 'Disney', 'The Walt Disney Company'),
                   'NFLX': ('Netflix, Inc.', 'Netflix', 'Netflix'),
                   'T': ('AT&T Inc.', 'AT&T', 'AT&T'),
                   'VZ': ('Verizon Communications Inc.', 'Verizon', 'Verizon Communications')},
         'ORCL': {'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                  'CSCO': ('Cisco Systems, Inc.', 'Cisco', 'Cisco Systems'),
                  'INTC': ('Intel Corporation', 'Intel', 'Intel')},
         'IBM': {'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                 'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation'),
                 'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                 'CSCO': ('Cisco Systems, Inc.', 'Cisco', 'Cisco Systems'),
                 'INTC': ('Intel Corporation', 'Intel', 'Intel')},
         'C': {'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
               'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
               'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
               'WFC': ('Wells Fargo & Company', 'Wells Fargo', 'Wells Fargo'),
               'GS': ('The Goldman Sachs Group, Inc.', 'Goldman Sachs', 'Goldman Sachs')},
         'WFC': {'WFC': ('Wells Fargo & Company', 'Wells Fargo', 'Wells Fargo'),
                 'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                 'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                 'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                 'MS': ('Morgan Stanley', 'Morgan Stanley', 'Morgan Stanley')},
         'PM': {'PM': ('Philip Morris International Inc.', 'Philip Morris', 'Philip Morris International'),
                'MO': ('Altria Group, Inc.', 'Altria', 'Altria Group'),
                'KO': ('The Coca-Cola Company', 'Coca-Cola', 'The Coca-Cola Company'),
                'PEP': ('PepsiCo, Inc.', 'Pepsi', 'PepsiCo'),
                'PG': ('The Procter & Gamble Company', 'Procter & Gamble', 'Procter & Gamble')},
         'GS': {'GS': ('The Goldman Sachs Group, Inc.', 'Goldman Sachs', 'Goldman Sachs'),
                'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                'MS': ('Morgan Stanley', 'Morgan Stanley', 'Morgan Stanley')},
         'NFLX': {'NFLX': ('Netflix, Inc.', 'Netflix', 'Netflix'),
                  'GOOGL': ('Alphabet Inc.', 'Google', 'Alphabet Inc.'),
                  'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)'),
                  'DIS': ('The Walt Disney Company', 'Disney', 'The Walt Disney Company'),
                  'META': ('Facebook, Inc.', 'Facebook', 'Facebook, Inc.')},
         'BA': {'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                'LMT': ('Lockheed Martin Corporation', 'Lockheed Martin', 'Lockheed Martin'),
                'HON': ('Honeywell International Inc.', 'Honeywell', 'Honeywell'),
                'MMM': ('3M Company', '3M', '3M'),
                'CAT': ('Caterpillar Inc.', 'Caterpillar', 'Caterpillar Inc.')},
         'MCD': {'MCD': ("McDonald's Corporation", "McDonald's", "McDonald's"),
                 'YUM': ('Yum! Brands, Inc.', 'Yum Brands', 'Yum! Brands'),
                 'SBUX': ('Starbucks Corporation', 'Starbucks', 'Starbucks'),
                 'NKE': ('Nike, Inc.', 'Nike', 'Nike, Inc.'),
                 'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot')},
         'MMM': {'MMM': ('3M Company', '3M', '3M'),
                 'HON': ('Honeywell International Inc.', 'Honeywell', 'Honeywell'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                 'CAT': ('Caterpillar Inc.', 'Caterpillar', 'Caterpillar Inc.'),
                 'GE': ('General Electric Company', 'General Electric', 'General Electric')},
         'HON': {'HON': ('Honeywell International Inc.', 'Honeywell', 'Honeywell'),
                 'MMM': ('3M Company', '3M', '3M'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                 'CAT': ('Caterpillar Inc.', 'Caterpillar', 'Caterpillar Inc.'),
                 'LMT': ('Lockheed Martin Corporation', 'Lockheed Martin', 'Lockheed Martin')},
         'AXP': {'AXP': ('American Express Company', 'American Express', 'American Express'),
                 'V': ('Visa Inc.', 'Visa', 'Visa Inc.'),
                 'MA': ('Mastercard Incorporated', 'Mastercard', 'Mastercard'),
                 'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                 'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase')},
         'AVGO': {'AVGO': ('Broadcom Inc.', 'Broadcom', 'Broadcom Inc.'),
                  'CSCO': ('Cisco Systems, Inc.', 'Cisco', 'Cisco Systems'),
                  'INTC': ('Intel Corporation', 'Intel', 'Intel'),
                  'IBM': ('International Business Machines Corporation', 'IBM', 'IBM'),
                  'ORCL': ('Oracle Corporation', 'Oracle', 'Oracle Corporation')},
         'UNP': {'UNP': ('Union Pacific Corporation', 'Union Pacific', 'Union Pacific Railroad'),
                 'CSX': ('CSX Corporation', 'CSX', 'CSX Corporation'),
                 'NSC': ('Norfolk Southern Corporation', 'Norfolk Southern', 'Norfolk Southern'),
                 'UPS': ('United Parcel Service, Inc.', 'UPS', 'United Parcel Service'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing')},
         'UPS': {'UPS': ('United Parcel Service, Inc.', 'UPS', 'United Parcel Service'),
                 'FDX': ('FedEx Corporation', 'FedEx', 'FedEx'),
                 'UNP': ('Union Pacific Corporation', 'Union Pacific', 'Union Pacific Railroad'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                 'CAT': ('Caterpillar Inc.', 'Caterpillar', 'Caterpillar Inc.')},
         'ABBV': {'ABBV': ('AbbVie Inc.', 'AbbVie', 'AbbVie'),
                  'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                  'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer'),
                  'MRK': ('Merck & Co., Inc.', 'Merck', 'Merck & Co.'),
                  'ABT': ('Abbott Laboratories', 'Abbott', 'Abbott Laboratories')},
         'CAT': {'CAT': ('Caterpillar Inc.', 'Caterpillar', 'Caterpillar Inc.'),
                 'DE': ('Deere & Company', 'John Deere', 'Deere & Company'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                 'MMM': ('3M Company', '3M', '3M'),
                 'HON': ('Honeywell International Inc.', 'Honeywell', 'Honeywell')},
         'MS': {'MS': ('Morgan Stanley', 'Morgan Stanley', 'Morgan Stanley'),
                'GS': ('The Goldman Sachs Group, Inc.', 'Goldman Sachs', 'Goldman Sachs'),
                'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                'WFC': ('Wells Fargo & Company', 'Wells Fargo', 'Wells Fargo')},
         'CVS': {'CVS': ('CVS Health Corporation', 'CVS Health', 'CVS Health'),
                 'WBA': ('Walgreens Boots Alliance, Inc.', 'Walgreens', 'Walgreens Boots Alliance'),
                 'UNH': ('UnitedHealth Group Incorporated', 'UnitedHealth', 'UnitedHealth Group'),
                 'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'PFE': ('Pfizer Inc.', 'Pfizer', 'Pfizer')},
         'UNH': {'UNH': ('UnitedHealth Group Incorporated', 'UnitedHealth', 'UnitedHealth Group'),
                 'HUM': ('Humana Inc.', 'Humana', 'Humana'),
                 'CVS': ('CVS Health Corporation', 'CVS Health', 'CVS Health'),
                 'JNJ': ('Johnson & Johnson', 'Johnson & Johnson', 'Johnson & Johnson'),
                 'ELV': ('Elevance Health Inc.', 'Anthem', 'Elevance Health')},
         'LMT': {'LMT': ('Lockheed Martin Corporation', 'Lockheed Martin', 'Lockheed Martin'),
                 'BA': ('The Boeing Company', 'Boeing', 'Boeing'),
                 'GD': ('General Dynamics Corporation', 'General Dynamics', 'General Dynamics'),
                 'NOC': ('Northrop Grumman Corporation', 'Northrop Grumman', 'Northrop Grumman'),
                 'HON': ('Honeywell International Inc.', 'Honeywell', 'Honeywell')},
         'TGT': {'TGT': ('Target Corporation', 'Target', 'Target Corporation'),
                 'WMT': ('Walmart Inc.', 'Walmart', 'Walmart'),
                 'AMZN': ('Amazon.com, Inc.', 'Amazon', 'Amazon (company)'),
                 'COST': ('Costco Wholesale Corporation', 'Costco', 'Costco'),
                 'HD': ('The Home Depot, Inc.', 'Home Depot', 'The Home Depot')},
         'META': {'GOOGL': ('Alphabet Inc.', 'Google', 'Alphabet Inc.'),
                  'AAPL': ('Apple Inc.', 'Apple', 'Apple Inc.'),
                  'MSFT': ('Microsoft Corporation', 'Microsoft', 'Microsoft'),
                  'NFLX': ('Netflix, Inc.', 'Netflix', 'Netflix'),
                  'META': ('Facebook, Inc.', 'Facebook', 'Facebook, Inc.')},
         'BRK-B': {'JPM': ('JPMorgan Chase & Co.', 'JPMorgan Chase', 'JPMorgan Chase'),
                   'BAC': ('Bank of America Corporation', 'Bank of America', 'Bank of America'),
                   'C': ('Citigroup Inc.', 'Citigroup', 'Citigroup'),
                   'GS': ('The Goldman Sachs Group, Inc.', 'Goldman Sachs', 'Goldman Sachs'),
                   'BRK-B': ('Berkshire Hathaway Inc.', 'Berkshire Hathaway', 'Berkshire Hathaway')}}

## Prices

In [4]:
df_download = yf.download(stocks + ['SPY'], start='2015-12-01', end='2020-01-01', auto_adjust=True, progress=False)
df_prices = df_download['Close'][stocks + ['SPY']].add_suffix('_price')
df_prices.index.name = 'date'
df_prices

Ticker,AAPL_price,MSFT_price,GOOGL_price,AMZN_price,META_price,BRK-B_price,JNJ_price,JPM_price,XOM_price,BAC_price,...,UNP_price,UPS_price,ABBV_price,CAT_price,MS_price,CVS_price,UNH_price,LMT_price,TGT_price,SPY_price
date,,,,,,,,,,,,,,,,,,,,,
2015-12-01,26.384714,48.046402,38.820091,33.952999,106.103760,136.429993,76.370476,50.612267,51.804806,14.039582,...,66.459152,69.123253,37.809643,55.669060,26.140072,69.195801,96.709747,165.583664,52.660389,176.127304
2015-12-02,26.146362,48.037701,38.525902,33.800499,105.063728,135.270004,76.139191,49.901115,50.324486,13.928911,...,64.632339,68.408058,36.976826,55.256744,25.858429,68.507080,98.280396,164.636185,52.023926,174.329910
2015-12-03,25.903515,47.158905,38.047951,33.312500,103.389755,132.660004,75.199089,49.257324,49.603313,13.675947,...,62.364513,67.739204,35.951832,54.175407,25.532322,67.310944,96.074844,162.019348,52.935219,171.888840
2015-12-04,26.764713,48.646763,38.593258,33.632000,105.172691,136.479996,76.810669,50.821869,49.887978,14.071203,...,61.663712,68.381584,36.630894,54.439903,26.177114,69.203026,98.363922,164.222580,53.448730,175.241135
2015-12-07,26.596069,48.559765,38.285194,33.491501,104.608093,135.600006,77.086723,50.155624,48.584797,13.865671,...,60.860512,67.845161,36.047932,53.179657,25.702789,69.840996,96.835098,164.779037,53.079872,174.179474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,68.402298,148.487335,66.587860,89.460503,203.174042,225.470001,121.305222,114.395409,51.953457,29.918219,...,155.127777,88.938599,68.021568,129.391937,41.272182,60.012215,261.914612,325.672424,106.730637,292.082001
2019-12-26,69.759422,149.704407,67.481369,93.438499,205.818726,226.449997,121.222137,115.609383,52.035069,30.173061,...,155.921097,89.472244,68.006462,130.041214,41.614159,59.988026,262.901642,327.450256,107.012924,293.636780
2019-12-27,69.732941,149.978073,67.093559,93.489998,206.125793,226.139999,121.155617,115.692528,51.857018,30.028643,...,156.438538,89.397072,67.529480,130.093842,41.597870,59.923599,263.186218,328.535370,107.278625,293.564209


In [5]:
missing_prices = df_prices.columns[df_prices.isna().any()].tolist()
missing_prices

[]

In [6]:
df_prices.to_csv('data/prices.csv')

## Wikipedia pages

In [7]:
headers = {'User-Agent': "MyWikipediaApp/1.0 (your@email.here)"}

pages = {}
for ticker in comps:
    for comp_ticker, (official_name, colloquial_name, page_title) in comps[ticker].items():
        pages[comp_ticker] = page_title

len(pages)

72

In [8]:
resolved_pages = {}
for ticker, page_title in pages.items():
    redirect = requests.get('https://en.wikipedia.org/w/api.php',
                            headers = headers,
                            params  = {'action'    : 'query',
                                       'titles'    : page_title,
                                       'redirects' : 1,
                                       'format'    : 'json'}).json()
    resolved_pages[ticker] = list(redirect['query']['pages'].values())[0]['title']
    time.sleep(0.1)

{ticker: (pages[ticker], resolved_pages[ticker]) for ticker in pages if pages[ticker] != resolved_pages[ticker]}

{'META': ('Facebook, Inc.', 'Meta Platforms'),
 'HD': ('The Home Depot', 'Home Depot'),
 'VZ': ('Verizon Communications', 'Verizon'),
 'CSCO': ('Cisco Systems', 'Cisco'),
 'AVGO': ('Broadcom Inc.', 'Broadcom'),
 'MO': ('Altria Group', 'Altria'),
 'NSC': ('Norfolk Southern', 'Norfolk Southern Railway'),
 'DE': ('Deere & Company', 'John Deere')}

## Redirects

In [9]:
page_titles = {}
for ticker, page_title in tqdm(resolved_pages.items()):
    page_titles[ticker] = [page_title]
    params = {'action'    : 'query',
              'titles'    : page_title,
              'prop'      : 'redirects',
              'rdlimit'   : 'max',
              'format'    : 'json'}
    while True:
        redirects = requests.get('https://en.wikipedia.org/w/api.php', headers=headers, params=params).json()
        page_titles[ticker] += [redirect['title'] for page in redirects['query']['pages'].values() for redirect in page.get('redirects', [])]
        if 'continue' not in redirects:
            break
        params.update(redirects['continue'])
    time.sleep(0.1)

pd.Series({ticker: len(titles) for ticker, titles in page_titles.items()}).describe()

  0%|          | 0/72 [00:00<?, ?it/s]

  1%|▏         | 1/72 [00:00<00:25,  2.76it/s]

  3%|▎         | 2/72 [00:00<00:25,  2.74it/s]

  4%|▍         | 3/72 [00:01<00:24,  2.76it/s]

  6%|▌         | 4/72 [00:01<00:25,  2.71it/s]

...

 97%|█████████▋| 70/72 [00:24<00:00,  2.88it/s]

 99%|█████████▊| 71/72 [00:25<00:00,  2.88it/s]

100%|██████████| 72/72 [00:25<00:00,  2.94it/s]

100%|██████████| 72/72 [00:25<00:00,  2.84it/s]

count     72.000000
mean      40.652778
std       24.812627
min        6.000000
25%       25.000000
50%       34.500000
75%       47.000000
max      144.000000
dtype: float64

## Wikipedia pageviews

In [10]:
api_url = ('https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia.org/'
           'all-access/all-agents/PAGE_ID/daily/2015120100/2019123100')

session = requests.Session()
views = {}
failed_pages = []
for ticker, titles in tqdm(page_titles.items()):
    page_views = []
    for page_title in titles:
        page_id = requests.utils.quote(page_title.replace(' ', '_'), safe='')
        for attempt in range(5):
            response = session.get(api_url.replace('PAGE_ID', page_id), headers=headers)
            if response.status_code in [200, 404]:
                break
            time.sleep(5)
        if response.status_code == 200:
            df = pd.DataFrame(response.json()['items'])
            df['date'] = pd.to_datetime(df.timestamp.str[:-2])
            page_views.append(df.set_index('date').views)
        elif response.status_code != 404:
            failed_pages.append((ticker, page_title, response.status_code))
        time.sleep(0.02)
    if page_views:
        views[f'{ticker}_views'] = pd.concat(page_views, axis=1).sum(axis=1)
    else:
        failed_pages.append((ticker, resolved_pages[ticker], 'no views'))

failed_pages

  0%|          | 0/72 [00:00<?, ?it/s]

  1%|▏         | 1/72 [00:35<41:35, 35.14s/it]

  3%|▎         | 2/72 [01:02<35:33, 30.48s/it]

  4%|▍         | 3/72 [01:13<24:59, 21.73s/it]

  6%|▌         | 4/72 [01:36<25:06, 22.15s/it]

...

 97%|█████████▋| 70/72 [18:01<00:20, 10.39s/it]

 99%|█████████▊| 71/72 [18:12<00:10, 10.51s/it]

100%|██████████| 72/72 [18:32<00:00, 13.50s/it]

100%|██████████| 72/72 [18:32<00:00, 15.46s/it]

[]

In [11]:
df_views = pd.DataFrame(views).sort_index()
df_views.index.name = 'date'
df_views

,AAPL_views,MSFT_views,GOOGL_views,IBM_views,ORCL_views,NFLX_views,META_views,AMZN_views,HD_views,TGT_views,...,FDX_views,DE_views,CVS_views,WBA_views,UNH_views,HUM_views,ELV_views,GD_views,NOC_views,BRK-B_views
date,,,,,,,,,,,,,,,,,,,,,
2015-12-01,16070.0,9372.0,6040.0,6059.0,3947.0,11055.0,104.0,14806.0,1730.0,3825.0,...,1917.0,1463.0,563.0,457.0,1513.0,418.0,756.0,1433.0,1371.0,5288.0
2015-12-02,16713.0,10063.0,6572.0,6158.0,4310.0,11054.0,158.0,13740.0,1752.0,3308.0,...,2053.0,1502.0,590.0,452.0,1674.0,384.0,654.0,1399.0,1320.0,7248.0
2015-12-03,16173.0,9371.0,5957.0,6232.0,3941.0,10506.0,114.0,13183.0,1481.0,3078.0,...,1829.0,1442.0,485.0,435.0,1597.0,337.0,750.0,1412.0,1385.0,5696.0
2015-12-04,17248.0,13383.0,5548.0,5468.0,3658.0,10053.0,128.0,13141.0,1648.0,2690.0,...,1622.0,1298.0,494.0,450.0,1642.0,366.0,679.0,1302.0,1238.0,5043.0
2015-12-05,11864.0,6699.0,3853.0,4157.0,2548.0,10044.0,78.0,11622.0,1272.0,2428.0,...,1347.0,933.0,331.0,314.0,815.0,214.0,370.0,1231.0,1072.0,4045.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-27,11789.0,7577.0,6782.0,4271.0,2901.0,18856.0,5503.0,15580.0,2406.0,2804.0,...,2604.0,1470.0,593.0,920.0,1000.0,313.0,760.0,1259.0,2543.0,6348.0
2019-12-28,11499.0,6874.0,6125.0,4611.0,2711.0,18286.0,5682.0,14381.0,2297.0,2675.0,...,1635.0,1375.0,609.0,724.0,828.0,223.0,441.0,1113.0,1400.0,5460.0
2019-12-29,11292.0,7997.0,5859.0,4582.0,2767.0,18579.0,5840.0,15908.0,2094.0,2630.0,...,1594.0,1387.0,705.0,763.0,1035.0,289.0,504.0,984.0,1148.0,5470.0


In [12]:
df_views.isna().sum()[df_views.isna().sum() > 0]

Series([], dtype: int64)

In [13]:
df_views.to_csv('data/views.csv')